In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
from scipy.stats import wasserstein_distance, linregress
from IPython.display import display

In [ ]:
TASK = Path.cwd().parent
ROOT = TASK.parent
INPUT, OUTPUT = TASK/'input', TASK/'output'
DIAGNOSTICS = OUTPUT
CONFIG = json.loads((ROOT/'config.json').read_text())
Y, SEED = CONFIG['last_year'], CONFIG['seed']
MODELS = CONFIG['models']

In [ ]:
panel = pd.read_csv(DIAGNOSTICS/'simulation/fit/data/panel_complete.csv',index_col='dblp_id')
observed = panel.to_numpy(dtype=float).T
models = {source:np.load(DIAGNOSTICS/'simulation'/source/'trajectories.npy',mmap_mode='r') for source in MODELS}
generated_from = (pd.read_csv(DIAGNOSTICS/'simulation_metadata.csv')
                  .set_index('model')['generated_from_age'].astype(int))

In [ ]:
# exact W1 on the raw annual output samples, including zeros
rows = []
for source,array in models.items():
    for age in range(Y+1):
        rows.append({'source':source,'age':age,'n_empirical':observed.shape[1],'n_simulated':array.shape[1],
                     'generated_from_age':generated_from[source],'is_generated':age>=generated_from[source],
                     'wasserstein1':wasserstein_distance(observed[age],array[age])})
distances = pd.DataFrame(rows)
distances.to_csv(OUTPUT/'annual_wasserstein.csv',index=False)

In [ ]:
# equally weighted mean and raw-scale OLS trend over all ages
rows = []
for source,frame in distances.groupby('source',sort=False):
    fit = linregress(frame.age,frame.wasserstein1)
    rows.append({'source':source,'start_age':int(frame.age.min()),'end_age':int(frame.age.max()),
                 'n_years':len(frame),'n_initialized_years':int((~frame.is_generated).sum()),
                 'mean_wasserstein1':frame.wasserstein1.mean(),'slope_wasserstein1':fit.slope})
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT/'annual_wasserstein_summary.csv',index=False)
display(summary)